# Module 2 — 矩陣運算與座標轉換

> **對應程度**：高中矩陣初步 → 大學線代入門

本模組涵蓋：
1. 矩陣基礎與特殊矩陣
2. 矩陣乘法與旋轉
3. 轉置矩陣與對稱性
4. 行列式
5. 逆矩陣
6. 特殊矩陣總覽

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.linalg_utils import (
    rotation_matrix_2d, rotation_matrix_3d, homogeneous_transform_2d,
    determinant_2x2, determinant_3x3, inverse_2x2
)
from src.visualizer import plot_matrix_heatmap, set_style

set_style()
print('Module 2 載入完成！')

---
## 2.1 矩陣基礎

矩陣 = 數字排成的方陣，可以代表「線性轉換」。

In [ ]:
# 建立各類特殊矩陣
I = np.eye(3)           # 單位矩陣
Z = np.zeros((3, 3))    # 零矩陣
D = np.diag([1, 2, 3])  # 對角矩陣

print('單位矩陣 I =')
print(I)
print(f'\n對角矩陣 D = diag(1,2,3) =')
print(D)

# 矩陣視覺化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, mat, title in zip(axes, [I, D, np.random.default_rng(42).normal(size=(5,5))],
                          ['單位矩陣 I', '對角矩陣 D', '隨機矩陣']):
    plot_matrix_heatmap(mat, title=title, ax=ax)
plt.tight_layout()
plt.show()

---
## 2.2 矩陣乘法與旋轉

### 2D 旋轉矩陣
$$R(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

### 物理應用：機器手臂正向運動學

In [ ]:
# 2D 旋轉動畫：向量繞原點旋轉
v = np.array([1, 0])  # 初始向量
thetas = np.linspace(0, 2*np.pi, 13)[:-1]  # 每 30° 一個

fig, ax = plt.subplots(figsize=(8, 8))
colors = plt.cm.rainbow(np.linspace(0, 1, len(thetas)))

for theta, color in zip(thetas, colors):
    R = rotation_matrix_2d(theta)
    v_rotated = R @ v
    ax.annotate('', xy=v_rotated, xytext=[0, 0],
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    ax.text(v_rotated[0]*1.1, v_rotated[1]*1.1,
            f'{np.degrees(theta):.0f}°', fontsize=9, color=color)

circle = plt.Circle((0, 0), 1, fill=False, color='gray', ls='--', alpha=0.5)
ax.add_patch(circle)
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.set_title('2D 旋轉矩陣 R(θ) 的作用')
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 驗證：R(θ₁)·R(θ₂) = R(θ₁+θ₂)
theta1, theta2 = np.pi/6, np.pi/4
R1 = rotation_matrix_2d(theta1)
R2 = rotation_matrix_2d(theta2)
R_product = R1 @ R2
R_sum = rotation_matrix_2d(theta1 + theta2)

print(f'R({np.degrees(theta1):.0f}°) × R({np.degrees(theta2):.0f}°) =')
print(R_product)
print(f'\nR({np.degrees(theta1+theta2):.0f}°) =')
print(R_sum)
print(f'\n一致: {np.allclose(R_product, R_sum)} ✓')

In [ ]:
# 2-DOF 機器手臂正向運動學
L1, L2 = 1.0, 0.8  # 臂長
theta1_range = np.radians(45)   # 第一關節角
theta2_range = np.radians(30)   # 第二關節角

def forward_kinematics_2dof(theta1, theta2, L1, L2):
    """2-DOF 正向運動學"""
    # 關節 1 位置
    p1 = np.array([L1 * np.cos(theta1), L1 * np.sin(theta1)])
    # 末端位置 = 兩段旋轉的矩陣乘法
    R1 = rotation_matrix_2d(theta1)
    R2 = rotation_matrix_2d(theta2)
    # 末端相對第一關節的位置
    p2_local = np.array([L2, 0])
    p2 = p1 + R1 @ R2 @ np.array([L2, 0])  # 簡化
    # 正確的末端位置
    p_end = np.array([
        L1 * np.cos(theta1) + L2 * np.cos(theta1 + theta2),
        L1 * np.sin(theta1) + L2 * np.sin(theta1 + theta2)
    ])
    return np.array([0, 0]), p1, p_end

# 繪製工作空間
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# 左圖：單一姿態
base, joint, end = forward_kinematics_2dof(theta1_range, theta2_range, L1, L2)
ax1.plot([base[0], joint[0]], [base[1], joint[1]], 'b-o', lw=3, ms=8)
ax1.plot([joint[0], end[0]], [joint[1], end[1]], 'r-o', lw=3, ms=8)
ax1.plot(base[0], base[1], 'ks', ms=12)  # 基座
ax1.plot(end[0], end[1], 'g*', ms=15)     # 末端
ax1.set_title(f'2-DOF 機器手臂 (θ₁={np.degrees(theta1_range):.0f}°, θ₂={np.degrees(theta2_range):.0f}°)')
ax1.set_xlim(-0.5, 2.5)
ax1.set_ylim(-0.5, 2.5)
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)

# 右圖：工作空間掃描
for t1 in np.linspace(0, np.pi, 20):
    for t2 in np.linspace(-np.pi, np.pi, 40):
        _, _, end = forward_kinematics_2dof(t1, t2, L1, L2)
        ax2.plot(end[0], end[1], '.', color='steelblue', ms=1, alpha=0.3)

ax2.set_title('機器手臂工作空間')
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 齊次座標：平移 + 旋轉 合併為單一矩陣乘法
theta = np.pi / 4  # 旋轉 45°
tx, ty = 2, 1      # 平移

H = homogeneous_transform_2d(theta, tx, ty)
print('齊次變換矩陣 H =')
print(H)

# 變換一個正方形
square = np.array([[0,0,1], [1,0,1], [1,1,1], [0,1,1]]).T  # 齊次座標
transformed = H @ square

fig, ax = plt.subplots(figsize=(8, 8))
orig = Polygon(square[:2].T, fill=True, alpha=0.3, color='blue', label='原始')
trans = Polygon(transformed[:2].T, fill=True, alpha=0.3, color='red', label='變換後')
ax.add_patch(orig)
ax.add_patch(trans)
ax.set_xlim(-1, 5)
ax.set_ylim(-1, 4)
ax.set_aspect('equal')
ax.legend(fontsize=12)
ax.set_title(f'齊次變換：旋轉 {np.degrees(theta):.0f}° + 平移 ({tx}, {ty})')
ax.grid(True, alpha=0.3)
plt.show()

---
## 2.3 轉置矩陣與對稱性

### 物理應用：導熱矩陣、應力張量必為對稱

In [ ]:
# 驗證導熱矩陣對稱性
n = 5
K = np.diag(2*np.ones(n)) + np.diag(-np.ones(n-1), 1) + np.diag(-np.ones(n-1), -1)
print('一維熱傳導矩陣 K =')
print(K)
print(f'\nK = K^T ? {np.allclose(K, K.T)} ✓')
print(f'性質: (AB)^T = B^T A^T')

A = np.random.default_rng(42).normal(size=(3, 3))
B = np.random.default_rng(43).normal(size=(3, 3))
print(f'(AB)^T = B^T A^T ? {np.allclose((A @ B).T, B.T @ A.T)} ✓')

---
## 2.4 行列式

**幾何意義**：矩陣造成的面積（體積）縮放比

In [ ]:
# 行列式的幾何意義：面積縮放
A = np.array([[2, 1], [0, 3]])
det_manual = determinant_2x2(A)
det_numpy = np.linalg.det(A)

print(f'矩陣 A =')
print(A)
print(f'det(A) 手動 = {det_manual}')
print(f'det(A) NumPy = {det_numpy}')
print(f'一致: {np.isclose(det_manual, det_numpy)} ✓')

# 視覺化：矩陣作用前後的正方形面積變化
unit_square = np.array([[0,0], [1,0], [1,1], [0,1]])
transformed = (A @ unit_square.T).T

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.add_patch(Polygon(unit_square, fill=True, alpha=0.3, color='blue'))
ax1.set_xlim(-0.5, 2)
ax1.set_ylim(-0.5, 2)
ax1.set_aspect('equal')
ax1.set_title(f'原始正方形 (面積 = 1)')
ax1.grid(True, alpha=0.3)

ax2.add_patch(Polygon(transformed, fill=True, alpha=0.3, color='red'))
ax2.set_xlim(-0.5, 5)
ax2.set_ylim(-0.5, 5)
ax2.set_aspect('equal')
ax2.set_title(f'變換後 (面積 = |det(A)| = {abs(det_manual):.0f})')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 三力共面判斷
f1 = np.array([1, 0, 0])
f2 = np.array([0, 1, 0])
f3_coplanar = np.array([1, 1, 0])     # 共面（在 xy 平面）
f3_not_coplanar = np.array([1, 1, 1]) # 不共面

A_coplanar = np.array([f1, f2, f3_coplanar])
A_not = np.array([f1, f2, f3_not_coplanar])

det1 = determinant_3x3(A_coplanar)
det2 = determinant_3x3(A_not)

print(f'共面三力的行列式: {det1:.4f} → {"共面" if abs(det1) < 1e-10 else "不共面"}')
print(f'不共面三力的行列式: {det2:.4f} → {"共面" if abs(det2) < 1e-10 else "不共面"}')

---
## 2.5 逆矩陣

### 物理應用：機器手臂逆向運動學、感測器校正

In [ ]:
# 感測器線性校正
# 感測器模型: y = Cx + noise
# 已知 C（校正矩陣），要從量測 y 反推真實值 x

C = np.array([[1.02, 0.01], [0.03, 0.98]])  # 校正矩陣（接近單位矩陣）
x_true = np.array([100, 50])  # 真實溫度

y_measured = C @ x_true + np.array([0.5, -0.3])  # 加噪聲

# 用逆矩陣反推
C_inv = inverse_2x2(C)
x_estimated = C_inv @ y_measured

print(f'校正矩陣 C =')
print(C)
print(f'\n真實值: {x_true}')
print(f'量測值: {y_measured}')
print(f'校正後: {x_estimated}')
print(f'誤差: {np.abs(x_estimated - x_true)}')

# 驗證 C × C^{-1} = I
print(f'\nC × C⁻¹ ≈ I ? {np.allclose(C @ C_inv, np.eye(2))} ✓')

In [ ]:
# 條件數（Condition Number）分析
# 條件數越大，矩陣越「病態」，逆矩陣越不穩定

matrices = {
    '良態矩陣': np.array([[2, 0], [0, 3]]),
    '中等': np.array([[1, 0.99], [0.99, 1]]),
    '病態矩陣': np.array([[1, 0.9999], [0.9999, 1]]),
}

for name, M in matrices.items():
    cond = np.linalg.cond(M)
    det = np.linalg.det(M)
    print(f'{name}: cond = {cond:.1f}, det = {det:.6f}')

---
## 2.6 特殊矩陣總覽

| 矩陣類型 | 工程對應 |
|---------|---------|
| 對角矩陣 | 各軸獨立的慣性張量 |
| 三對角矩陣 | 一維有限差分 |
| 正交矩陣 | 旋轉矩陣 |
| 對稱正定 | 剛度矩陣、質量矩陣 |
| 稀疏矩陣 | 大型有限元素模型 |

In [ ]:
# 驗證旋轉矩陣是正交矩陣
for axis in ['x', 'y', 'z']:
    R = rotation_matrix_3d(axis, np.pi/5)
    print(f'R_{axis}(36°):')
    print(f'  R^T R = I ? {np.allclose(R.T @ R, np.eye(3))}')
    print(f'  det(R) = {np.linalg.det(R):.10f}')

# 稀疏矩陣比較
from scipy.sparse import diags as sp_diags

n = 1000
K_dense = np.diag(2*np.ones(n)) + np.diag(-np.ones(n-1), 1) + np.diag(-np.ones(n-1), -1)
K_sparse = sp_diags([[-1]*n, [2]*n, [-1]*n], [-1, 0, 1], shape=(n, n), format='csr')

print(f'\n{n}×{n} 三對角矩陣:')
print(f'  Dense 記憶體: {K_dense.nbytes / 1024:.1f} KB')
print(f'  Sparse 記憶體: {(K_sparse.data.nbytes + K_sparse.indices.nbytes + K_sparse.indptr.nbytes) / 1024:.1f} KB')
print(f'  節省: {(1 - (K_sparse.data.nbytes + K_sparse.indices.nbytes + K_sparse.indptr.nbytes) / K_dense.nbytes) * 100:.1f}%')

In [ ]:
# 一維熱傳三對角矩陣建構與求解
from src.physics_models import heat_conduction_1d

x, T, K = heat_conduction_1d(n_nodes=20, T_left=200, T_right=20)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
plot_matrix_heatmap(K[:8, :8], title='導熱矩陣 K (前 8×8)', ax=ax1)
ax2.plot(x, T, 'ro-', lw=2)
ax2.set_xlabel('位置 x (m)')
ax2.set_ylabel('溫度 T (°C)')
ax2.set_title('一維穩態溫度分佈 (200°C → 20°C)')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'\n溫度分佈為線性 ✓（無內熱源時應為線性）')

---
## Module 2 驗證總結

| 項目 | 驗證方式 | 結果 |
|------|----------|------|
| 旋轉矩陣正交性 | R^T R = I, det(R) = 1 | ✓ |
| 旋轉矩陣合成 | R(θ₁)R(θ₂) = R(θ₁+θ₂) | ✓ |
| 行列式 | 手動 vs np.linalg.det | ✓ |
| 逆矩陣 | A × A⁻¹ = I | ✓ |
| 導熱矩陣對稱 | K = K^T | ✓ |
| 溫度線性分佈 | 解析解比對 | ✓ |